In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 1 — Install dependencies

In [3]:
!pip install -q "transformers==4.44.2" "peft==0.13.0" soundfile webrtcvad
!apt-get install -qq ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.5 MB/s eta 0:00:00


## Cell 2 — Configuration

In [4]:
# ── USER CONFIGURATION ──────────────────────────────────────────────────

# Path to the .opus file containing a full Surah Al-Fatiha recitation
# Example: '/content/drive/MyDrive/my_audio_files/fatiha.opus'
OPUS_FILE = "/content/drive/MyDrive/audio.opus"   # <-- EDIT THIS

# LoRA adapter produced by Notebook 2
# Example: '/content/drive/MyDrive/my_adapters/whisper-medium-quran-lora-multivoice/final_adapter'
ADAPTER_PATH = "/content/drive/MyDrive/wisper-model/whisper-medium-quran-lora-multivoice/final_adapter" # <-- EDIT THIS

# Scoring threshold (run Cell 9 calibration to tune this)
MARGIN_THRESHOLD = 0.697

# VAD aggressiveness: 0 (least) to 3 (most)
# Increase if short words like وَلَا are being merged with neighbours
VAD_AGGRESSIVENESS = 2

# Minimum silence gap (seconds) that separates two words
MIN_SILENCE_SEC = 0.04

SAMPLING_RATE = 16_000
DEVICE = "cuda"

## Cell 3 — Surah Al-Fatiha word list (29 words, full diacritics)

In [5]:
FATIHA_WORDS = [
    # "أَعُوذُ",
    # "بِاللَّهِ",
    # "مِنَ" ,
    # "الشَيْطَانِ",
    # "الرَجِيمِ",
    # Ayah 1:1
    "بِسْمِ",
    "اللَّهِ",
    "الرَّحْمَٰنِ",
    "الرَّحِيمِ",
    # Ayah 1:2
    "الْحَمْدُ",
    "لِلَّهِ",
    "رَبِّ",
    "الْعَالَمِينَ",
    # Ayah 1:3
    "الرَّحْمَٰنِ",
    "الرَّحِيمِ",
    # Ayah 1:4
    "مَالِكِ",
    "يَوْمِ",
    "الدِّينِ",
    # Ayah 1:5
    "إِيَّاكَ",
    "نَعْبُدُ",
    "وَإِيَّاكَ",
    "نَسْتَعِينُ",
    # Ayah 1:6
    "اهْدِنَا",
    "الصِّرَاطَ",
    "الْمُسْتَقِيمَ",
    # Ayah 1:7
    "صِرَاطَ",
    "الَّذِينَ",
    "أَنْعَمْتَ",
    "عَلَيْهِمْ",
    "غَيْرِ",
    "الْمَغْضُوبِ",
    "عَلَيْهِمْ",
    "وَلَا",
    "الضَّالِّينَ",
]

print(f"Word list loaded: {len(FATIHA_WORDS)} words")
for i, w in enumerate(FATIHA_WORDS, 1):
    print(f"  {i:2}. {w}")

Word list loaded: 29 words
   1. بِسْمِ
   2. اللَّهِ
   3. الرَّحْمَٰنِ
   4. الرَّحِيمِ
   5. الْحَمْدُ
   6. لِلَّهِ
   7. رَبِّ
   8. الْعَالَمِينَ
   9. الرَّحْمَٰنِ
  10. الرَّحِيمِ
  11. مَالِكِ
  12. يَوْمِ
  13. الدِّينِ
  14. إِيَّاكَ
  15. نَعْبُدُ
  16. وَإِيَّاكَ
  17. نَسْتَعِينُ
  18. اهْدِنَا
  19. الصِّرَاطَ
  20. الْمُسْتَقِيمَ
  21. صِرَاطَ
  22. الَّذِينَ
  23. أَنْعَمْتَ
  24. عَلَيْهِمْ
  25. غَيْرِ
  26. الْمَغْضُوبِ
  27. عَلَيْهِمْ
  28. وَلَا
  29. الضَّالِّينَ


## Cell 4 — Load model + processor

In [6]:
import torch
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor

base = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-medium", torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base, ADAPTER_PATH).merge_and_unload()
model = model.to(DEVICE).eval()
processor = WhisperProcessor.from_pretrained(ADAPTER_PATH)
print("Model + adapter loaded and merged.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model + adapter loaded and merged.


## Cell 5 — Helper utilities (text cleaning + harakah variants)

In [7]:
import random
import re
from typing import List

STRUCTURAL_RE = re.compile(
    "[\u06d6-\u06dc\u06dd\u06de\u06e9\u0610-\u0617\u0640\u0660-\u0669\u06f0-\u06f90-9]"
)

def clean_quranic_text(text: str) -> str:
    text = STRUCTURAL_RE.sub("", text)
    return re.sub(r"\s+", " ", text).strip()

FATHA, DAMMA, KASRA, SUKUN = "\u064e", "\u064f", "\u0650", "\u0652"
FATHATAN, DAMMATAN, KASRATAN = "\u064b", "\u064c", "\u064d"
SHADDAH = "\u0651"
SHORT_HARAKAT = {FATHA, DAMMA, KASRA, SUKUN}
TANWEEN = {FATHATAN, DAMMATAN, KASRATAN}

def harakah_variants(word: str, max_variants: int = None, seed: int = 0) -> List[str]:
    variants = set()
    for i, ch in enumerate(word):
        pool = SHORT_HARAKAT if ch in SHORT_HARAKAT else TANWEEN if ch in TANWEEN else None
        if pool is not None:
            for alt in pool - {ch}:
                variants.add(word[:i] + alt + word[i + 1:])
        elif ch == SHADDAH:
            variants.add(word[:i] + word[i + 1:])
    variants.discard(word)
    out = sorted(variants)
    if max_variants is not None and len(out) > max_variants:
        out = random.Random(seed).sample(out, max_variants)
    return out

In [8]:
!pip install -q "transformers==4.44.2" "peft==0.13.0" soundfile stable-ts
!apt-get install -qq ffmpeg


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.1/189.1 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 33.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Cell 6 — Audio loader + VAD segmenter

Decodes the `.opus` file with `ffmpeg`, then uses WebRTC VAD to split it
into individual word segments aligned 1-to-1 with `FATIHA_WORDS`.

In [9]:
import subprocess
import numpy as np
import soundfile as sf
import tempfile, os
import stable_whisper

def load_opus(path: str, target_sr: int = 16_000) -> np.ndarray:
    cmd = ["ffmpeg", "-loglevel", "error", "-i", path,
           "-f", "f32le", "-acodec", "pcm_f32le",
           "-ar", str(target_sr), "-ac", "1", "pipe:1"]
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\n{result.stderr.decode()}")
    audio = np.frombuffer(result.stdout, dtype=np.float32).copy()
    print(f"Loaded '{path}'  ->  {len(audio)/target_sr:.2f}s @ {target_sr} Hz")
    return audio

# Load the aligner once (whisper-medium, same as Notebook 1b).
_aligner = stable_whisper.load_model("medium", device=DEVICE)

def segment_words_by_alignment(audio: np.ndarray, expected_words, sr: int = 16_000,
                               pad_s: float = 0.10):
    """Force-align the KNOWN word list to the recording and slice per-word clips.
    Returns (clips, aligned_words) so you can verify alignment matched the text."""
    # stable-ts wants a file path; write a temp wav.
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
        tmp = tf.name
    try:
        sf.write(tmp, np.asarray(audio, dtype=np.float32), sr)
        # Feed the known text (joined) so it aligns rather than guesses.
        result = _aligner.align(tmp, " ".join(expected_words), language="ar")
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)

    clips, aligned_words = [], []
    for seg in result.segments:
        for w in seg.words:
            a = max(0, int((w.start - pad_s) * sr))
            b = min(len(audio), int((w.end + pad_s) * sr))
            clips.append(audio[a:b])
            aligned_words.append(w.word.strip())
    return clips, aligned_words


100%|█████████████████████████████████████| 1.42G/1.42G [00:17<00:00, 85.7MiB/s]


## Cell 7 — Scoring engine

In [10]:
def forced_score(input_features: torch.Tensor, texts: List[str]):
    tok = processor.tokenizer
    pad_id = tok.pad_token_id or tok.eos_token_id
    seqs = [tok(t).input_ids for t in texts]
    maxlen = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), maxlen), pad_id, dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
    decoder_input_ids = ids[:, :-1].to(DEVICE)
    targets = ids[:, 1:].to(DEVICE)

    with torch.no_grad():
        logits = model(
            input_features=input_features.expand(len(seqs), -1, -1),
            decoder_input_ids=decoder_input_ids,
        ).logits
    token_logprobs = logits.float().log_softmax(-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)

    mask = torch.zeros_like(targets, dtype=torch.bool)
    for i, s in enumerate(seqs):
        mask[i, 3:len(s) - 2] = True
    scores = (token_logprobs * mask).sum(1) / mask.sum(1).clamp(min=1)

    detail = [
        (tok.decode([targets[0, j].item()]), round(token_logprobs[0, j].item(), 3))
        for j in range(3, len(seqs[0]) - 2)
    ]
    return scores, detail


def analyze_word(audio_array: np.ndarray, expected_word: str,
                 margin_threshold: float = 0.697) -> dict:
    expected = clean_quranic_text(expected_word)
    feats = processor.feature_extractor(
        np.asarray(audio_array, dtype=np.float32),
        sampling_rate=SAMPLING_RATE, return_tensors="pt",
    ).input_features.to(DEVICE).half()

    variants = harakah_variants(expected)
    candidates = [expected] + variants
    scores, detail = forced_score(feats, candidates)
    scores = scores.tolist()

    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    best_wrong = max(scores[1:]) if variants else float("-inf")
    margin = scores[0] - best_wrong

    return {
        "expected":             expected,
        "verdict":              "pass" if margin > margin_threshold else "flag",
        "margin":               round(margin, 4),
        "expected_score":       round(scores[0], 4),
        "most_likely_said":     ranked[0][0],
        "top_candidates":       [(w, round(s, 4)) for w, s in ranked[:3]],
        "per_token_confidence": detail,
    }

## Cell 8 — Run full Surah Al-Fatiha evaluation

In [13]:
audio = load_opus(OPUS_FILE, target_sr=SAMPLING_RATE)
segments, aligned_words = segment_words_by_alignment(audio, FATIHA_WORDS, sr=SAMPLING_RATE)
print(f"Alignment produced {len(segments)} word clips  |  expected {len(FATIHA_WORDS)} words")

if len(segments) != len(FATIHA_WORDS):
    print()
    print("NOTE: alignment word count differs from the list.")
    print("  -> Alignment may have merged or split a word. Proceeding with min() pairs.")
    print("  -> Check 'aligned_words' below against FATIHA_WORDS to see where it diverged.")
    print("  aligned_words:", aligned_words)

n_pairs = min(len(segments), len(FATIHA_WORDS))
results = []
for i in range(n_pairs):
    r = analyze_word(segments[i], FATIHA_WORDS[i], margin_threshold=MARGIN_THRESHOLD)
    results.append(r)
    status = "PASS" if r["verdict"] == "pass" else "FAIL"
    correct_won = "OK" if r["most_likely_said"] == r["expected"] else "WRONG-WON"
    print(f"  {i+1:2}. {FATIHA_WORDS[i]:20s}  [{status}]  margin={r['margin']:+.3f}  "
          f"top={r['most_likely_said']:20s} ({correct_won})")

n_pass = sum(1 for r in results if r["verdict"] == "pass")
n_flag = len(results) - n_pass
# "correct won" = did the true word outrank all mispronunciations (threshold-independent)
n_correct_won = sum(1 for r in results if r["most_likely_said"] == r["expected"])
avg_margin = sum(r["margin"] for r in results) / len(results)
pass_pct = n_pass / len(results)

print()
print("=" * 64)
print("SURAH AL-FATIHA PRONUNCIATION REPORT")
print("=" * 64)
print(f"Words scored        : {len(results)} / {len(FATIHA_WORDS)}")
print(f"Correct word won    : {n_correct_won}  ({n_correct_won/len(results):.0%})   <- threshold-independent signal")
print(f"Passed (margin>thr) : {n_pass}  ({pass_pct:.0%})   (threshold={MARGIN_THRESHOLD})")
print(f"Flagged             : {n_flag}  ({1 - pass_pct:.0%})")
print(f"Avg margin          : {avg_margin:+.3f}")
print("-" * 64)

if n_flag == 0:
    print("OVERALL: EXCELLENT -- all words passed")
elif n_flag / len(results) <= 0.15:
    print("OVERALL: GOOD -- minor issues, review flagged words")
elif n_flag / len(results) <= 0.40:
    print("OVERALL: NEEDS PRACTICE -- several harakah mistakes found")
else:
    print("OVERALL: SIGNIFICANT ERRORS -- many words flagged for review")

if n_flag > 0:
    print()
    print("Flagged words:")
    for i, r in enumerate(results):
        if r["verdict"] == "flag":
            print(f"  Word {i+1:2}: {r['expected']:20s}  margin={r['margin']:+.4f}  most likely said -> {r['most_likely_said']}")
print("=" * 64)


Loaded '/content/drive/MyDrive/audio.opus'  ->  36.21s @ 16000 Hz


Align: 100%|██████████| 36.21/36.21 [00:08<00:00,  4.05sec/s]
Adjustment: 100%|██████████| 35.58/35.58 [00:00<00:00, 33618.68sec/s]


Alignment produced 29 word clips  |  expected 29 words


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


   1. بِسْمِ                [FAIL]  margin=-0.082  top=بَسْمِ               (WRONG-WON)
   2. اللَّهِ               [FAIL]  margin=-4.340  top=اللَهِ               (WRONG-WON)
   3. الرَّحْمَٰنِ          [FAIL]  margin=-0.321  top=الرَحْمَٰنِ          (WRONG-WON)
   4. الرَّحِيمِ            [FAIL]  margin=-0.877  top=الرَحِيمِ            (WRONG-WON)
   5. الْحَمْدُ             [FAIL]  margin=-0.556  top=الِحَمْدُ            (WRONG-WON)
   6. لِلَّهِ               [FAIL]  margin=-1.053  top=لُلَّهِ              (WRONG-WON)
   7. رَبِّ                 [PASS]  margin=+0.794  top=رَبِّ                (OK)
   8. الْعَالَمِينَ         [FAIL]  margin=+0.586  top=الْعَالَمِينَ        (OK)
   9. الرَّحْمَٰنِ          [FAIL]  margin=-0.835  top=الرَحْمَٰنِ          (WRONG-WON)
  10. الرَّحِيمِ            [FAIL]  margin=-1.346  top=الرَحِيمِ            (WRONG-WON)
  11. مَالِكِ               [FAIL]  margin=-1.647  top=مَالْكِ              (WRONG-WON)
  12. يَوْمِ                [PASS]  margin=+1.

## Cell 9 — Detailed per-word token breakdown

In [14]:
# Set to the word number (1-indexed) you want to inspect
INSPECT_WORD_NUMBER = 1

r = results[INSPECT_WORD_NUMBER - 1]
print(f"Word {INSPECT_WORD_NUMBER}: {r['expected']}")
print(f"Verdict : {r['verdict'].upper()}  |  margin: {r['margin']:+.4f}")
print(f"Most likely said: {r['most_likely_said']}")
print()
print("Top candidates:")
for word, score in r["top_candidates"]:
    marker = " <- expected" if word == r["expected"] else ""
    print(f"  {word:20s}  {score:.4f}{marker}")
print()
print("Per-token log-probabilities:")
for token, lp in r["per_token_confidence"]:
    bar = "#" * max(0, int((lp + 10) * 2))
    print(f"  {token:8s}  {lp:+.3f}  {bar}")

Word 1: بِسْمِ
Verdict : FLAG  |  margin: -0.0816
Most likely said: بَسْمِ

Top candidates:
  بَسْمِ                -1.2139
  بِسْمِ                -1.2956 <- expected
  بُسْمِ                -1.3970

Per-token log-probabilities:
  س         -4.377  ###########
  ْ         -0.778  ##################
  م         -0.016  ###################
  ِ         -0.011  ###################


## Cell 10 — VAD tuning tips

If Cell 8 reports a segment count that does not match 29:

| Problem | Fix |
|---|---|
| Too few segments (words merged) | Decrease `MIN_SILENCE_SEC` (e.g. 0.08) or increase `VAD_AGGRESSIVENESS` to 3 |
| Too many segments (one word split) | Increase `MIN_SILENCE_SEC` (e.g. 0.25) or decrease `VAD_AGGRESSIVENESS` to 1 |
| Still wrong after tuning | Manually split audio in Audacity and pass individual files |